In [1]:
import torch
import numpy as np


In [2]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

get_device = get_device()
get_device

device(type='cuda')

In [3]:
import pandas as pd

try: # KAGGLE
    df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
except: 
    try: # COLAB
        df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
    except: # LOCAL
        df = pd.read_csv('../data/train.csv')
        test_df = pd.read_csv('../data/test.csv')
        
df.shape, test_df.shape

((2000, 8), (500, 7))

# Config

In [4]:
MODEL_NAME = "google-bert/bert-base-uncased"
OPTIONS = ["A", "B", "C", "D", "E"]
NUM_CHOICES = len(OPTIONS)
BATCH_SIZE = 100
EPOCHS = 20

In [5]:
# changing the labels


---

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {opt: i for i, opt in enumerate(OPTIONS)}
id2label = {v: k for k, v in label2id.items()}

df['labels'] = df['answer'].apply(lambda x: label2id[x])
df = df.drop(columns=['id','answer'], axis=1)
test_df = test_df.drop(columns=['id'], axis=1)
# test_df['labels'] = test_df['answer'].apply(lambda x: label2id[x])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
test_df.columns, df.columns

(Index(['prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object'),
 Index(['prompt', 'A', 'B', 'C', 'D', 'E', 'labels'], dtype='object'))

# Preprocess function

In [8]:
label2id

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [9]:
def map_labels(string):
    label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    return label2id[string]

def preprocess_function(examples):
    prompts = [[context] * 5 for context in examples["prompt"]]
    choices = [
        [examples[choice][i] for choice in ["A", "B", "C", "D", "E"]] 
        for i in range(len(examples["prompt"]))
    ]

    prompts = sum(prompts, [])
    choices = sum(choices, [])
    
    tokenized_examples = tokenizer(prompts, choices, truncation=True, padding="max_length")     
    
    output = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)]
        for k, v in tokenized_examples.items()
    }

    if 'labels' in list(examples.keys()):
        output['labels'] = examples['labels']

    return output 
    
from typing import Any, Optional
from dataclasses import dataclass
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: Any
    padding: bool = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "labels"
        labels = []
        
        # Extract and convert labels
        if label_name in features[0].keys():
            raw_labels = [feature.pop(label_name) for feature in features]
            letter_to_id = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
            for l in raw_labels:
                if isinstance(l, str) and l in letter_to_id:
                    labels.append(letter_to_id[l])
                else:
                    labels.append(int(l))
        else:
            labels = None
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] 
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return batch

# Dataset load

In [10]:
from datasets import Dataset


full_dataset = Dataset.from_pandas(df)
split = full_dataset.train_test_split(test_size=0.2, seed=42)
# print(split)

encoded_train = split['train'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['train'].column_names if col != "answer"]
)

encoded_val = split['test'].map(
    preprocess_function,
    batched=True,
    remove_columns=[col for col in split['test'].column_names if col != "answer"]
)

dt = Dataset.from_pandas(test_df)
encoded_test = dt.map(
    preprocess_function,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=dt.column_names,
)

del full_dataset, split, dt

encoded_train.shape, encoded_test.shape, encoded_val.shape

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

((1600, 4), (500, 3), (400, 4))

In [11]:
encoded_train, encoded_val, encoded_test

(Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 1600
 }),
 Dataset({
     features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 400
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 500
 }))

In [12]:
import numpy as np

def map3_score(labels, logits):
    # Ensure logits is a numpy array
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    # Ensure logits is 2D: (batch_size, num_choices)
    if logits.ndim == 1:
        logits = logits.reshape(-1, 5)  # Reshape assuming 5 choices (A, B, C, D, E)

    top3 = np.argsort(-logits, axis=1)[:, :3]
    scores = []
    
    for lab, ranked in zip(labels, top3):
        if lab == ranked[0]:
            scores.append(1.0)
        elif lab == ranked[1]:
            scores.append(0.5)
        elif lab == ranked[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)
            
    return float(np.mean(scores))

def compute_metrics(eval_pred) -> dict:
    logits, labels = eval_pred  
    
    return {
        "map3": map3_score(labels, logits)
    }

# Load Model

In [13]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# wandb init

In [15]:
import wandb 
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

IS_WANDB = wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


# Training

In [16]:
training_args = TrainingArguments(
    output_dir="./google-wala-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="map3",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    logging_steps=50,
    run_name="google-wala-model",
)

wandb.init(project="24f3004148-t22026")

data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)
trainer.train()

wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Test predictions & Submission

In [110]:
preds = trainer.predict(encoded_test)
raw_logits = preds.predictions
top_3_indices = torch.topk(torch.tensor(raw_logits), k=3, dim=-1).indices.numpy()
raw_logits

array([[ 1.8875734 ,  1.9059879 ,  1.7573143 ,  1.8644106 ,  1.8335801 ],
       [ 0.94080603,  1.2611746 ,  0.94308645,  0.97928107, -0.7617043 ],
       [-2.6476622 ,  1.8487818 , -2.6406887 , -2.5406678 , -1.9733847 ],
       ...,
       [-2.4536097 , -2.351853  , -2.4611127 , -2.4895246 , -2.4685764 ],
       [-2.4781673 , -2.3347876 , -2.4749334 , -2.4768732 ,  2.0554352 ],
       [-2.6787388 , -2.5644548 ,  1.6672806 , -2.5702384 , -2.6133869 ]],
      dtype=float32)

In [113]:
top_3_predictions = [
    " ".join([id2label[idx] for idx in row])
    for row in top_3_indices
]

submission_df = pd.DataFrame({
    'ID': range(1, len(test_df)+1),
    'Prediction': top_3_predictions
})
submission_df.to_csv("submission.csv", index=False)

In [114]:
submission_df

,ID,Prediction
0,1,B A D
1,2,B D C
2,3,B E D
3,4,E C A
4,5,C A D
...,...,...
495,496,A E B
496,497,C B D
497,498,B A C
498,499,E B C
